# Sailing Race Data Visualization (Plotly)

This notebook explores sailor/team/venue performance patterns using race results.
It builds summary metrics (Points + Ratio) and visualizes how performance varies by team and venue.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

## Load Data + Create Core Metrics

In [ ]:
df_races = pd.read_csv("races.csv",converters={"Teams": lambda x: [y.strip().split("'")[1] for y in x.strip("[]").split(", ")]})

df_races['Points'] = df_races['Teams'].apply(len) - df_races['Score'] + 1
df_races['Ratio'] = 1 - (df_races['Score'] / df_races['Teams'].apply(len))
df_races["Score"] = df_races["Score"].astype(int)
df_races["Ratio"] = df_races["Ratio"].astype(float)
df_races.head(3)

## Build Sailor- and Team-Level Aggregates

In [ ]:
# Summarize performance per sailor and per team (mean Score / Points / Ratio + regatta count).
df_avg = pd.DataFrame(columns=[])

overallAvg = {"Points": df_races.groupby("Position")['Points'].mean()[1],"Ratio": df_races.groupby("Position")['Ratio'].mean()[1]}

for p in df_races['Sailor'].unique():
  ser = pd.Series()
  ser['Sailor'] = p
  ser['mean_Score'] = df_races.loc[df_races['Sailor']==p]['Score'].mean()
  ser['mean_Points'] = df_races.loc[df_races['Sailor']==p]['Points'].mean()
  ser['mean_Ratio'] = float(df_races.loc[df_races['Sailor']==p]['Ratio'].mean())
  ser['regattas'] = int(len(df_races.loc[df_races['Sailor']==p]['Regatta'].unique()))
  if len(df_races.loc[df_races['Sailor']==p]['Position'].unique()) == 0:
    ser['position'] = "Unknown"
  else:
    ser['position'] = df_races.loc[df_races['Sailor']==p]['Position'].unique()[0]
  ser['team'] = df_races.loc[df_races['Sailor'] == p]['Team'].iloc[0]
  df_avg = pd.concat([df_avg, ser.to_frame().T], ignore_index=True)

df_team_avg = pd.DataFrame(columns=["team", 'ratio'])
for team in set(df_avg['team']):
  temp_Series = pd.Series()
  temp_Series['team'] = team
  temp_Series['ratio'] = df_avg.loc[df_avg['team'] == team]['mean_Ratio'].mean()
  df_team_avg = pd.concat([df_team_avg, temp_Series.to_frame().T])

df_avg['mean_Score'] = df_avg['mean_Score'].astype(float)
df_avg['mean_Points'] = df_avg['mean_Points'].astype(float)
df_avg['mean_Ratio'] = df_avg['mean_Ratio'].astype(float)

## Venue-Level Aggregates (Sailor × Venue)

In [ ]:
# Compute sailor-level performance metrics by venue.
df_venue = pd.DataFrame()
for p in df_races['Sailor'].unique():
  for v in df_races.loc[df_races['Sailor'] == p]['Venue'].unique():
    ser = pd.Series()
    ser['Sailor'] = p
    ser['venue'] = v
    ser['mean_Score'] = df_races.loc[(df_races['Sailor']==p) & (df_races['Venue'] == v)]['Score'].mean()
    ser['mean_Points'] = df_races.loc[(df_races['Sailor']==p) & (df_races['Venue'] == v)]['Points'].mean()
    ser['mean_Ratio'] = float(df_races.loc[(df_races['Sailor']==p) & (df_races['Venue'] == v)]['Ratio'].mean())
    ser['regattas'] = int(len(df_races.loc[(df_races['Sailor']==p) & (df_races['Venue'] == v)]['Regatta'].unique()))
    if len(df_races.loc[(df_races['Sailor']==p) & (df_races['Venue'] == v)]['Position'].unique()) == 0:
      ser['position'] = "Unknown"
    else:
      ser['position'] = df_races.loc[(df_races['Sailor']==p) & (df_races['Venue'] == v)]['Position'].unique()[0]
    ser['team'] = df_races.loc[df_races['Sailor'] == p]['Team'].iloc[0]
    df_venue = pd.concat([df_venue, ser.to_frame().T], ignore_index=True)
    
df_venue['mean_Score'] = df_venue['mean_Score'].astype(float)
df_venue['mean_Points'] = df_venue['mean_Points'].astype(float)
df_venue['mean_Ratio'] = df_venue['mean_Ratio'].astype(float)

## Visualizations

In [ ]:
fig=px.scatter(df_avg, x='team', y='mean_Ratio', color='mean_Ratio', color_continuous_scale=["red", "gray", "lime"], category_orders={'team': df_team_avg.sort_values('ratio', ascending=False)['team']}, hover_name='Sailor')

fig.add_trace(go.Scatter(
  x=[df_team_avg.sort_values('ratio', ascending=False)['team'].iloc[0], df_team_avg.sort_values('ratio', ascending=False)['team'].tail(1).iloc[0]], 
  y=[overallAvg['Ratio'], overallAvg['Ratio']], 
  mode='lines', 
  name='Average Skipper Ratio'))

fig.update_layout(height=600, width=3000,  title='Mean Ratio per Sailor (grouped by team)',
    xaxis_title='Team (Sorted by avg mean)',
    yaxis_title='Mean Ratio (Per sailor)')
fig.show()

### Northeastern: Sailor-Level Performance

In [ ]:
fig=px.scatter(df_avg.loc[df_avg['team'] == "Northeastern"], x='mean_Points', y='mean_Ratio', color='mean_Ratio', color_continuous_scale=["red", "gray", "lime"], hover_name='Sailor')
fig.show()

### Venue Performance (All Sailors)


In [ ]:
fig=px.scatter(df_venue, x='venue', y='mean_Ratio', color='mean_Ratio', color_continuous_scale=["red", "gray", "lime"], hover_name='Sailor', hover_data=['team', 'regattas'])
fig.update_layout(height=600, width=2000)
fig.show()

### Deep Dive: UC San Diego (Skippers)

In [ ]:
fig=px.bar(df_venue.loc[(df_venue['venue']=="UC San Diego") & (df_venue['position'] == "Skipper")].sort_values('mean_Ratio', ascending=False),x='Sailor', y='mean_Ratio', hover_data=['team'], color='team')
fig.update_layout(xaxis_categoryorder = 'total descending')
fig.show()

### Average Performance by Team


In [ ]:
fig=px.bar(df_team_avg.sort_values('ratio', ascending=False),x='team', y='ratio')
fig.update_layout(height=600, width=3000)
fig.show()

### Top Sailors (Minimum 3 Regattas)


In [ ]:
fig = px.bar(df_avg.loc[df_avg['regattas'] > 2].sort_values('mean_Ratio', ascending=False).head(50),x='Sailor', y='mean_Ratio', hover_data='regattas', color='team')
fig.update_layout(xaxis_categoryorder = 'total descending')
fig.show()

### Example Team Snapshot: Tulane


In [ ]:
fig = px.bar(df_avg.loc[df_avg['team'] == 'Tulane'].sort_values('mean_Ratio', ascending=False),x='Sailor', y='mean_Ratio', hover_data='regattas')
fig.show()

### Northeastern Skippers: Ranking + Experience vs Performance


In [ ]:
fig = px.bar(df_avg.loc[(df_avg['team'] == 'Northeastern') & (df_avg['position'] == "Skipper")].sort_values('mean_Ratio', ascending=False),x='Sailor', y='mean_Ratio', hover_data='regattas')
fig.show()
fig = px.scatter(df_avg.loc[(df_avg['team'] == 'Northeastern') & (df_avg['position'] == "Skipper")].sort_values('mean_Ratio', ascending=False),x='regattas', y='mean_Ratio', hover_data='Sailor')
fig.show()

### Northeastern Crew: Ranking


In [ ]:
fig = px.bar(df_avg.loc[(df_avg['team'] == 'Northeastern') & (df_avg['position'] == "Crew")].sort_values('mean_Ratio', ascending=False),x='Sailor', y='mean_Ratio', hover_data='regattas')
fig.show()

## Key Takeaways

- The Ratio metric makes performance comparable across races with different fleet sizes.
- Team averages show meaningful separation, not just individual outliers.
- Venue-specific results vary, suggesting location/context influences outcomes.
- Regatta count provides a basic experience proxy when comparing sailors.
